# Workflows de Negocio: CRM, Email y Slack Bot

Implementación en Python de los 4 workflows de negocio más rentables
para desplegar en n8n con Claude.

**Workflows cubiertos:**
1. Triaje de emails con respuesta borrador
2. Actualización automática de CRM desde emails
3. Slack Bot para preguntas internas
4. Informe semanal automático

In [ ]:
import anthropic
import json
from datetime import datetime, date

client = anthropic.Anthropic()
print("Cliente Anthropic listo")

## Workflow 1: Triaje de emails con respuesta borrador

Gmail recibe email → Claude clasifica y genera borrador → se guarda en Gmail para revisión humana

In [ ]:
EMAILS_NEGOCIO = [
    {
        "from": "pedro.sanchez@techcorp.es",
        "subject": "Problema con integración de API - URGENTE",
        "body": "Buenos días, llevamos 2 horas sin poder conectar con vuestra API. El error es 503. Tenemos un cliente esperando en producción. ¿Podéis ayudarnos urgentemente?",
        "id": "email_001"
    },
    {
        "from": "maria.costa@inversiones.com",
        "subject": "Interés en partnership estratégico",
        "body": "Hola, soy María Costa, directora de alianzas en InversionesCo. Gestionamos un portfolio de 50 empresas tecnológicas y creemos que podría haber sinergias muy interesantes. ¿Podríamos agendar una llamada?",
        "id": "email_002"
    },
    {
        "from": "noreply@ofertas-spam.com",
        "subject": "¡¡GANA DINERO FÁCIL!! Oferta exclusiva",
        "body": "Haz clic aquí para ganar 500€ al día sin esfuerzo...",
        "id": "email_003"
    }
]

def triar_email_con_borrador(email: dict) -> dict:
    """Clasificar email y generar borrador de respuesta."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        system="""Eres el asistente de comunicaciones de la empresa. Clasificas emails
y generas respuestas en tono profesional y cercano.
IMPORTANTE: Los borradores siempre son revisados por un humano antes de enviar.""",
        messages=[{
            "role": "user",
            "content": f"""Analiza este email y genera un borrador de respuesta.

DE: {email['from']}
ASUNTO: {email['subject']}
CUERPO: {email['body'][:1500]}

Responde en JSON:
{{
  "categoria": "soporte|comercial|facturacion|partnership|spam|otro",
  "prioridad": "alta|media|baja",
  "sentimiento": "positivo|neutro|negativo|urgente",
  "resumen_3_palabras": "...",
  "requiere_respuesta": true,
  "borrador_respuesta": "texto completo del borrador con saludo y despedida",
  "acciones_adicionales": []
}}"""
        }]
    )
    
    analisis = json.loads(response.content[0].text)
    analisis["email_id"] = email["id"]
    analisis["from"] = email["from"]
    analisis["procesado_en"] = datetime.now().isoformat()
    return analisis

print("Procesando emails de bandeja de entrada:\n")
for email in EMAILS_NEGOCIO:
    analisis = triar_email_con_borrador(email)
    print(f"{'='*55}")
    print(f"De: {email['from']}")
    print(f"Categoría: {analisis['categoria']} | Prioridad: {analisis['prioridad']}")
    print(f"Resumen: {analisis['resumen_3_palabras']}")
    if analisis['requiere_respuesta'] and analisis['categoria'] != 'spam':
        print(f"\nBORRADOR:\n{analisis['borrador_respuesta'][:300]}...")
    else:
        print("[No requiere respuesta - marcar como spam]")

## Workflow 2: Actualización de CRM desde emails

In [ ]:
# CRM simulado (en n8n sería el nodo HubSpot)
CRM_HUBSPOT = {
    "contactos": []
}

def extraer_datos_contacto_de_email(email: dict) -> dict:
    """Extrae datos del contacto del email para actualizar HubSpot."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"""Extrae datos del contacto de este email. JSON sin markdown:
{{
  "nombre": "...",
  "apellidos": "...",
  "email": "...",
  "empresa": "...",
  "cargo": "... o null",
  "interes_principal": "descripción breve de qué quiere",
  "etapa_ciclo": "prospect|lead_calificado|oportunidad|cliente|churned"
}}

De: {email['from']}
Asunto: {email['subject']}
Cuerpo: {email['body'][:800]}"""
        }]
    )
    
    contacto = json.loads(response.content[0].text)
    contacto["nota_crm"] = f"[Auto] Email: {email['subject']}. Interés: {contacto['interes_principal']}"
    contacto["actualizado_en"] = datetime.now().isoformat()
    return contacto

def actualizar_crm(contacto: dict):
    """Simula el nodo HubSpot de n8n: buscar o crear contacto y añadir nota."""
    # Buscar si existe (en n8n: HubSpot node → Search Contacts)
    existente = next((c for c in CRM_HUBSPOT["contactos"] 
                     if c["email"] == contacto["email"]), None)
    
    if existente:
        existente.update(contacto)
        print(f"[HubSpot] ACTUALIZADO: {contacto['nombre']} ({contacto['empresa']})")
    else:
        CRM_HUBSPOT["contactos"].append(contacto)
        print(f"[HubSpot] CREADO: {contacto['nombre']} ({contacto['empresa']})")
    
    print(f"  Etapa: {contacto['etapa_ciclo']}")
    print(f"  Nota: {contacto['nota_crm']}")

# Procesar los emails comerciales
emails_con_contacto = [e for e in EMAILS_NEGOCIO if "spam" not in e["from"]]
print("Actualizando CRM desde emails:\n")
for email in emails_con_contacto:
    contacto = extraer_datos_contacto_de_email(email)
    actualizar_crm(contacto)
    print()

print(f"CRM total: {len(CRM_HUBSPOT['contactos'])} contactos")

## Workflow 3: Slack Bot de preguntas internas

In [ ]:
BASE_CONOCIMIENTO_EMPRESA = """
POLÍTICAS DE EMPRESA:
- Vacaciones: 23 días laborables al año + festivos locales
- Horario: flexible con núcleo 10h-15h
- Home office: hasta 3 días/semana con acuerdo con manager
- Gastos: reembolso con ticket en < 30 días, límite 50€ sin aprobación previa
- Dietas viaje: 45€/día en España, 75€/día en el extranjero

HERRAMIENTAS:
- CRM: HubSpot (acceso todos los empleados)
- Gestión proyectos: Linear (tech), Notion (resto)
- Comunicación: Slack (trabajo) + email (externo)
- Docs: Google Workspace
- Contraseñas: 1Password (todos tienen licencia)

PROCESOS:
- Onboarding nuevos empleados: checklist en Notion > RRHH > Onboarding
- Solicitar equipo: formulario en #it-requests
- Baja médica: notificar a manager + RRHH en el mismo día
- Vacaciones: solicitar en Factorial con 2 semanas de antelación
- Gastos: subir tickets a Factorial en < 30 días tras el gasto
"""

def responder_pregunta_slack(pregunta: str, usuario: str = "empleado", canal: str = "#preguntas") -> dict:
    """Simula el Slack Bot: Webhook Trigger → Claude → respuesta en hilo."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        system="""Eres el asistente interno de la empresa. Respondes preguntas de empleados
basándote SOLO en la base de conocimiento proporcionada. Si no sabes la respuesta,
dilo claramente y sugiere a quién preguntar. Sé conciso (máximo 3 párrafos).
Usa formato Markdown con **negrita** para énfasis.""",
        messages=[{
            "role": "user",
            "content": f"BASE DE CONOCIMIENTO:\n{BASE_CONOCIMIENTO_EMPRESA}\n\nPREGUNTA: {pregunta}"
        }]
    )
    
    return {
        "canal": canal,
        "usuario": usuario,
        "pregunta": pregunta,
        "respuesta": response.content[0].text,
        "respondido_en": datetime.now().isoformat()
    }

# Simular preguntas de empleados en Slack
PREGUNTAS_EMPLEADOS = [
    "¿Cuántos días de vacaciones tengo al año?",
    "¿Dónde solicito el portátil nuevo?",
    "¿Cuál es el límite de gastos sin aprobación previa?",
    "¿Dónde está el checklist de onboarding para el nuevo compañero que empieza el lunes?"
]

print("SIMULACIÓN SLACK BOT\n" + "="*50)
for pregunta in PREGUNTAS_EMPLEADOS:
    respuesta = responder_pregunta_slack(pregunta)
    print(f"\n[#preguntas] ❓ {pregunta}")
    print(f"[Bot] {respuesta['respuesta'][:250]}..." if len(respuesta['respuesta']) > 250 else f"[Bot] {respuesta['respuesta']}")

## Workflow 4: Informe semanal automático

Cron trigger cada lunes a las 8:00 → recopila datos → Claude genera informe → envía por email

In [ ]:
# Simula los datos agregados de HubSpot + Linear + Google Analytics
# (en n8n serían nodos conectados a cada servicio)
DATOS_SEMANA = {
    "hubspot": {
        "nuevos_leads": 23,
        "oportunidades": 8,
        "cierres": 3,
        "mrr_nuevo": 4200
    },
    "linear": {
        "completadas": 47,
        "bugs_cerrados": 12,
        "incidencias_criticas": 1
    },
    "analytics": {
        "sesiones": 8420,
        "demos": 15,
        "conversion_pct": 0.18
    }
}

def generar_informe_semanal(datos: dict) -> dict:
    """Genera el informe ejecutivo semanal con Claude."""
    resumen_datos = f"""
VENTAS (semana):
- Nuevos leads: {datos['hubspot']['nuevos_leads']}
- Oportunidades activas: {datos['hubspot']['oportunidades']}
- Cierres: {datos['hubspot']['cierres']} ({datos['hubspot']['mrr_nuevo']}€ MRR nuevo)

PRODUCTO (semana):
- Tareas completadas: {datos['linear']['completadas']}
- Bugs cerrados: {datos['linear']['bugs_cerrados']}
- Incidencias críticas: {datos['linear']['incidencias_criticas']}

WEB (semana):
- Visitas: {datos['analytics']['sesiones']:,}
- Demos solicitadas: {datos['analytics']['demos']}
- Tasa de conversión: {datos['analytics']['conversion_pct']}%
"""
    
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        messages=[{
            "role": "user",
            "content": f"""Genera el informe semanal ejecutivo para el equipo directivo.

DATOS DE LA SEMANA:
{resumen_datos}

El informe debe incluir:
1. Resumen en 2 frases (qué fue bien, qué no)
2. 3 highlights positivos con datos
3. 2-3 áreas de atención o riesgo
4. 1 foco recomendado para esta semana

Tono: directo, sin florituras, orientado a acción."""
        }]
    )
    
    semana = date.today().strftime("%Y-W%V")
    return {
        "informe": response.content[0].text,
        "semana": semana,
        "generado_en": datetime.now().isoformat(),
        "datos_raw": resumen_datos
    }

print("Generando informe semanal...\n")
informe = generar_informe_semanal(DATOS_SEMANA)
print(f"INFORME SEMANAL — {informe['semana']}")
print("="*55)
print(informe["informe"])
print("\n[EMAIL enviado a: equipo-directivo@empresa.com]")

## Comparativa de modelos para cada workflow

Haiku es suficiente para todos estos casos. Veamos el coste.

In [ ]:
PRECIOS = {
    "claude-haiku-4-5-20251001": {"input": 0.80, "output": 4.00},
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00}
}

def calcular_coste_workflow(llamadas_dia: int, tokens_in: int, tokens_out: int, 
                             modelo: str = "claude-haiku-4-5-20251001") -> dict:
    p = PRECIOS[modelo]
    coste_llamada = (tokens_in * p["input"] + tokens_out * p["output"]) / 1_000_000
    return {
        "modelo": modelo,
        "coste_por_llamada_usd": round(coste_llamada, 6),
        "coste_dia_usd": round(coste_llamada * llamadas_dia, 4),
        "coste_mes_usd": round(coste_llamada * llamadas_dia * 30, 2)
    }

print("COSTE ESTIMADO DE LOS WORKFLOWS DE NEGOCIO")
print("="*55)
workflows = [
    ("Triaje emails", 50, 300, 400),
    ("Actualizar CRM", 50, 250, 250),
    ("Slack Bot", 30, 300, 300),
    ("Informe semanal", 1, 500, 500),
]

total_mes = 0
for nombre, llamadas, tok_in, tok_out in workflows:
    coste = calcular_coste_workflow(llamadas, tok_in, tok_out)
    total_mes += coste["coste_mes_usd"]
    print(f"\n{nombre}:")
    print(f"  {llamadas} llamadas/día × {coste['coste_por_llamada_usd']}$/llamada")
    print(f"  → {coste['coste_dia_usd']}$/día | {coste['coste_mes_usd']}$/mes")

print(f"\n{'='*55}")
print(f"TOTAL MENSUAL (Haiku): ${total_mes:.2f}/mes")
print(f"Con Sonnet sería: ${total_mes * 10:.2f}/mes (~10x más caro)")